In [ ]:
import pandas as pd
import numpy as np
import ast

In [ ]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
movies=movies.merge(credits,on='title')

In [ ]:
movies.shape

(4809, 23)

In [ ]:
movies=movies[["movie_id","title","overview","genres","keywords","cast","crew"]]

In [ ]:
import ast
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i["name"])
    return L

In [ ]:
movies.dropna(inplace=True) # Droping null values from the movies data frame

<ipython-input-9-f7d4e0491274>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies.dropna(inplace=True) # Droping null values from the movies data frame


In [ ]:
movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)
# The .apply() method is used to apply a function (in this case, convert) to each element in the column.
# This is a function that you have defined elsewhere in your code. The convert function determines how the values in the keywords column are transformed.

In [ ]:
movies.shape

(4806, 7)

In [ ]:
def convert3(obj):
    L=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter!=3:
            L.append(i["name"])
            counter+=1
        else:
            break
    return L

    # we only want top three caracters from the cast columns so that's why we are using COUNTER.

In [ ]:
movies["cast"]=movies["cast"].apply(convert3)

In [ ]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i["job"]=="Director":
            L.append(i["name"])
            break
    return L

In [ ]:
movies.shape

(4806, 7)

In [ ]:
movies["crew"]=movies["crew"].apply(fetch_director)

In [ ]:

# def collapse(L):
#     L1 = []
#     for i in L:
#         L1.append(i.replace(" ",""))
#     return L1

In [ ]:
# movies['cast'] = movies['cast'].apply(collapse)
# movies['crew'] = movies['crew'].apply(collapse)
# movies['genres'] = movies['genres'].apply(collapse)
# movies['keywords'] = movies['keywords'].apply(collapse)

# alternative of the below code

In [ ]:
movies["genres"]=movies["genres"].apply(lambda x:[i.replace(" ","") for i in x])
movies["keywords"]=movies["keywords"].apply(lambda x:[i.replace(" ","") for i in x])
movies["cast"]=movies["cast"].apply(lambda x:[i.replace(" ","") for i in x])
movies["crew"]=movies["crew"].apply(lambda x:[i.replace(" ","") for i in x])

# .apply(lambda x: ...): The apply function applies the lambda function to each element in the crew column.
# [i.replace(" ", "") for i in x]:
# This is a list comprehension that iterates over all elements (i) in x (assumed to be a list).
# For each i, it calls the replace method to remove spaces (" ") by replacing them with an empty string ("").
# The goal of the code is to remove all spaces from each string in the lists contained within the crew column.
# For example, if the crew column contains lists of crew member names, such as:

# Key Points:

# replace(" ", "") removes spaces from each string.
# The lambda function processes the entire list of names (x) in each row of the crew column.
# End Result: A column where all crew member names no longer contain spaces.

In [ ]:
movies["overview"]=movies["overview"].apply(lambda x:x.split())

In [ ]:
movies["tags"]=movies["overview"]+movies["genres"]+movies["keywords"]+ movies["cast"]+movies["crew"]

In [ ]:
movies.shape

(4806, 8)

In [ ]:
new_df=movies.drop(columns=['overview','genres','keywords','cast','crew'])

In [ ]:
new_df["tags"]=new_df["tags"].apply(lambda x:" ".join(x))

#  " ".join(x)
# The join method is a string method that concatenates all elements in a list into a single string, separated by the string specified before .join().
# In this case, " ".join(x) joins the elements of the list x using a space (" ") as the separator.

# x = ["Action", "Adventure", "Sci-Fi"]
# " ".join(x)  # Result: "Action Adventure Sci-Fi"


In [ ]:
new_df["tags"]=new_df["tags"].apply(lambda x:x.lower())

In [ ]:
new_df.shape

(4806, 3)

In [ ]:
# Steming

import nltk
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [ ]:
def stem(text):
    y = []  # Initialize an empty list to store stemmed words.
    for i in text.split():  # Split the input text into words using whitespace.
        y.append(ps.stem(i))  # Apply the stemming function `ps.stem()` on each word and add it to the list.
    return " ".join(y)  # Join the stemmed words back into a single string with spaces and return it.


In [ ]:
new_df["tags"]=new_df["tags"].apply(stem)

In [ ]:
# Vectorization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words="english")

In [ ]:
vectors=cv.fit_transform(new_df["tags"]).toarray()


In [ ]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors) # Calculate the cosine similarity matrix.
# similarity[1]# Get the shape of the similarity matrix.
 # Get the shape of the first row of the similarity matrix.

In [ ]:
new_df

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just want to play hi guitar and ca...
4805,72766,Newlyweds,a newlyw couple' honeymoon is upend by the arr...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduc a dedic q..."
4807,126186,Shanghai Calling,when ambiti new york attorney sam is sent to s...


In [ ]:
def recommend(movie):
    count=1
    movie_index=new_df[new_df["title"]==movie].index[0]
    distance=similarity[movie_index]
    movie_list=sorted(list(enumerate(distance)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movie_list:
      print(new_df.iloc[i[0]]["title"])
      #   movies=(new_df.iloc[i[0]]["title"],new_df.iloc[i[0]]["tags"])
      #   rec_movies=",".join(movies)
      #   print(f"{count}.{rec_movies}")
      #   count+=1
      # else:
      #   breakf count <=5:
      #

In [ ]:
recommend("Batman Begins")


The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


In [ ]:
import pickle

In [ ]:
pickle.dump(new_df,open("movies.pkl","wb"))

In [ ]:
pickle.dump(similarity,open("similarity.pkl","wb"))

In [ ]:
pickle.dump(new_df.to_dict(),open("movie_dict.pkl","wb"))